# v7e — Semi-Supervised Learning

## Problem
Getting more labeled data is expensive (requires expert FA engineers). Can we
leverage the patterns within our existing data more effectively?

## Approaches
1. **Self-Training**: Train on 50% labels, pseudo-label the rest, retrain
2. **Label Spreading**: Graph-based propagation through feature space
3. **Co-Training**: Two classifiers on different views teach each other

## Expected runtime: 5-15 min total

## Kernel: efaai_v3 (Python 3.12)

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack

warnings.filterwarnings("ignore")
np.random.seed(42)
rng = np.random.RandomState(42)

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.path.join(ROOT, ".."))

CSV = os.path.join(ROOT, "data", "v5a_eos_vs_noneos.csv")
TEXT_COL = "PSI Failure Desc"
LABEL_COL = "label"
print(f"ROOT: {ROOT}")

In [ ]:
# Load and split
df = pd.read_csv(CSV)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 3].reset_index(drop=True)
le = LabelEncoder()
df["y"] = le.fit_transform(df[LABEL_COL])
EOS_IDX = list(le.classes_).index("EOS")
X_text = df[TEXT_COL].values
y = df["y"].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(y_train)}, Test: {len(y_test)}")

In [ ]:
# TF-IDF features
tw = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=3000, sublinear_tf=True)
tc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=3000, sublinear_tf=True)

X_train_word = tw.fit_transform(X_train_text)
X_train_char = tc.fit_transform(X_train_text)
X_train_full = hstack([X_train_word, X_train_char])
X_test_word = tw.transform(X_test_text)
X_test_char = tc.transform(X_test_text)
X_test_full = hstack([X_test_word, X_test_char])

print(f"Features: {X_train_full.shape}")

## §3 — Baseline: Full Supervised

In [ ]:
# Baseline
print("Baseline: SMOTE + RF on full training data...")
sm = SMOTE(random_state=42)
Xr, yr = sm.fit_resample(X_train_full, y_train)
rf_full = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_full.fit(Xr, yr)
preds_full = rf_full.predict(X_test_full)

baseline_mf1 = f1_score(y_test, preds_full, average="macro")
baseline_acc = accuracy_score(y_test, preds_full)
baseline_ef1 = f1_score(y_test, preds_full, pos_label=EOS_IDX)
print(f"  Macro-F1: {baseline_mf1:.4f}, Acc: {baseline_acc:.4f}, EOS-F1: {baseline_ef1:.4f}")

all_results = [{"approach": "Baseline (Full Supervised)", "macro_f1": baseline_mf1, "accuracy": baseline_acc, "eos_f1": baseline_ef1}]

## §4 — Self-Training
Train on 50% labels, iteratively add high-confidence pseudo-labels.

In [ ]:
print("Self-Training (50% labeled)...")
t0 = time.time()

# Mask 50% as unlabeled
mask = rng.rand(len(y_train)) < 0.5
y_semi = y_train.copy()
y_semi[~mask] = -1
print(f"  Labeled: {mask.sum()}, Unlabeled: {(~mask).sum()}")

# Dense for SelfTrainingClassifier
X_train_dense = X_train_full.toarray()
X_test_dense = X_test_full.toarray()

base_rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
self_trainer = SelfTrainingClassifier(base_estimator=base_rf, threshold=0.85, max_iter=10, verbose=True)
self_trainer.fit(X_train_dense, y_semi)
preds_st = self_trainer.predict(X_test_dense)

st_mf1 = f1_score(y_test, preds_st, average="macro")
st_acc = accuracy_score(y_test, preds_st)
st_ef1 = f1_score(y_test, preds_st, pos_label=EOS_IDX)
print(f"\n  Self-Training: Macro-F1={st_mf1:.4f} (delta: {st_mf1-baseline_mf1:+.4f}), EOS-F1={st_ef1:.4f}")
print(f"  Time: {time.time()-t0:.1f}s")

all_results.append({"approach": "Self-Training (50% labeled)", "macro_f1": st_mf1, "accuracy": st_acc, "eos_f1": st_ef1})

## §5 — Label Spreading
Graph-based label propagation. Subsample to 5K for memory.

In [ ]:
print("Label Spreading (5K subsample)...")
t0 = time.time()

N_SUB = 5000
idx_sub = rng.choice(len(y_train), size=min(N_SUB, len(y_train)), replace=False)
X_sub = X_train_dense[idx_sub]
y_sub = y_train[idx_sub]

mask_sub = rng.rand(len(y_sub)) < 0.5
y_sub_semi = y_sub.copy()
y_sub_semi[~mask_sub] = -1
print(f"  Subsample: {len(y_sub)}, Labeled: {mask_sub.sum()}")

ls = LabelSpreading(kernel="knn", n_neighbors=10, max_iter=30, alpha=0.2)
ls.fit(X_sub, y_sub_semi)

# Accuracy of propagation
propagated = ls.transduction_
masked_acc = (propagated[~mask_sub] == y_sub[~mask_sub]).mean()
print(f"  Propagation accuracy: {masked_acc:.4f}")

# Train RF on propagated labels
rf_ls = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_ls.fit(X_sub, propagated)
preds_ls = rf_ls.predict(X_test_dense)

ls_mf1 = f1_score(y_test, preds_ls, average="macro")
ls_acc = accuracy_score(y_test, preds_ls)
ls_ef1 = f1_score(y_test, preds_ls, pos_label=EOS_IDX)
print(f"  Label Spreading: Macro-F1={ls_mf1:.4f} (delta: {ls_mf1-baseline_mf1:+.4f}), EOS-F1={ls_ef1:.4f}")
print(f"  Time: {time.time()-t0:.1f}s")

all_results.append({"approach": "Label Spreading (5K, 50% labeled)", "macro_f1": ls_mf1, "accuracy": ls_acc, "eos_f1": ls_ef1})

## §6 — Co-Training
Two classifiers (word view + char view) teach each other.

In [ ]:
print("Co-Training (word + char views)...")
t0 = time.time()

X_train_w = X_train_word.toarray()
X_train_c = X_train_char.toarray()
X_test_w = X_test_word.toarray()
X_test_c = X_test_char.toarray()

labeled_mask = rng.rand(len(y_train)) < 0.5
current_labeled = list(np.where(labeled_mask)[0])
current_unlabeled = list(np.where(~labeled_mask)[0])
print(f"  Initial labeled: {len(current_labeled)}, Unlabeled: {len(current_unlabeled)}")

N_ITER, TOP_K, CONF_THRESH = 5, 100, 0.90

for it in range(N_ITER):
    if not current_unlabeled:
        break
    clf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf_c = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf_w.fit(X_train_w[current_labeled], y_train[current_labeled])
    clf_c.fit(X_train_c[current_labeled], y_train[current_labeled])
    
    prob_w = clf_w.predict_proba(X_train_w[current_unlabeled])
    prob_c = clf_c.predict_proba(X_train_c[current_unlabeled])
    conf_w, conf_c = prob_w.max(axis=1), prob_c.max(axis=1)
    pred_w, pred_c = prob_w.argmax(axis=1), prob_c.argmax(axis=1)
    
    agree = (pred_w == pred_c) & (conf_w > CONF_THRESH) & (conf_c > CONF_THRESH)
    if agree.sum() == 0:
        agree = conf_w > CONF_THRESH  # fallback
    
    scores = conf_w + conf_c
    scores[~agree] = -1
    top_k = min(TOP_K, agree.sum())
    top_idx = np.argsort(scores)[-top_k:]
    
    new_labeled = [current_unlabeled[i] for i in top_idx]
    current_labeled.extend(new_labeled)
    current_unlabeled = [i for i in current_unlabeled if i not in new_labeled]
    print(f"  Iter {it+1}: +{len(new_labeled)} samples, labeled={len(current_labeled)}, remaining={len(current_unlabeled)}")

# Final model on expanded labeled set
sm_co = SMOTE(random_state=42)
X_co = X_train_full[current_labeled].toarray()
y_co = y_train[current_labeled]
Xr_co, yr_co = sm_co.fit_resample(X_co, y_co)
clf_final = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf_final.fit(Xr_co, yr_co)
preds_co = clf_final.predict(X_test_dense)

co_mf1 = f1_score(y_test, preds_co, average="macro")
co_acc = accuracy_score(y_test, preds_co)
co_ef1 = f1_score(y_test, preds_co, pos_label=EOS_IDX)
print(f"\n  Co-Training: Macro-F1={co_mf1:.4f} (delta: {co_mf1-baseline_mf1:+.4f}), EOS-F1={co_ef1:.4f}")
print(f"  Time: {time.time()-t0:.1f}s")

all_results.append({"approach": "Co-Training (word+char, 5 iters)", "macro_f1": co_mf1, "accuracy": co_acc, "eos_f1": co_ef1})

In [ ]:
# Summary
print("=" * 70)
print("  v7e SEMI-SUPERVISED LEARNING — SUMMARY")
print("=" * 70)

rdf = pd.DataFrame(all_results)
rdf["delta"] = rdf["macro_f1"] - baseline_mf1
rdf = rdf.sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(rdf[["approach", "macro_f1", "accuracy", "eos_f1", "delta"]].to_string(index=False))

best = rdf.iloc[0]
if best["delta"] > 0.005:
    print(f"\n\u2705 Best: {best['approach']} beats baseline by {best['delta']:+.4f}")
elif best["delta"] > -0.005:
    print(f"\n\u2248 Semi-supervised matches full supervision (similar with less labels!)")
else:
    print(f"\n\u26a0\ufe0f Semi-supervised underperforms. Our labeled data is well-utilized.")
    print("  These methods shine with truly unlabeled data (which we lack).")

rdf.to_csv(os.path.join(ROOT, "results", "v7e_semisupervised_results.csv"), index=False)
print("\nSaved: results/v7e_semisupervised_results.csv")
print("\n\u2705 v7e complete.")